In [1]:
import pickle
import numpy as np
import tensorflow
import keras

In [2]:
# loading the data from pickle :
with open(r"C:\Users\trupt\OneDrive\Desktop\Emotion Detection\data\images.p", 'rb') as f:
    x = pickle.load(f)

with open(r"C:\Users\trupt\OneDrive\Desktop\Emotion Detection\data\labels.p", 'rb') as f:
    y = pickle.load(f)


In [3]:
print ('Images:',x.shape)
print ('Labels:',y.shape)

Images: (16082, 48, 48)
Labels: (16082,)


In [4]:
# labelEncoding output
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)


In [5]:
y

array([0, 0, 0, ..., 4, 4, 4], shape=(16082,))

In [6]:
# Stabdardizing the images for CNN Model:
x = x/255
x=x.reshape(-1,48,48,1)   # -1 -> the total number of images in x variable, 1 -> Number of channel in gray scale, 48,48 -> image shape

In [7]:
# Using one-hot encoding to output column:
from keras.utils import to_categorical
y = to_categorical(y)   # Don't run it multiple times.

In [8]:
# Splitting Training and testing datas    ( IF THE MODEL NOT GIVES ABOVE 50% ACCURACY RE-RUN FROM THIS SHELL ONCE AGAIN)
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.1)  #0.1 -> 10% data for testing

In [9]:
print("xtrain:",xtrain.shape)
print("xtest:",xtest.shape)
print("ytrain:",ytrain.shape)
print("ytest:",ytest.shape)

xtrain: (14473, 48, 48, 1)
xtest: (1609, 48, 48, 1)
ytrain: (14473, 5)
ytest: (1609, 5)


In [10]:
# CNN Model training :
from keras.models import Sequential
from keras.optimizers import Adam
from keras.layers import Conv2D,MaxPooling2D,Flatten,Dense


In [11]:
model = Sequential()
# 1st Convolutional Layer:
model.add(Conv2D(50, (3, 3), input_shape=(48, 48, 1), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 2nd Convolutional layer:
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 3rd Convolutional layer:
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Adding flatten Layer: converts 2D to 1D
model.add(Flatten())

# Hidden Layer:
model.add(Dense(256, activation='relu'))  # 256 number of neurones

# Output Layer:
model.add(Dense(5, activation='softmax'))
model.compile(Adam(learning_rate=0.001),loss = 'categorical_crossentropy', metrics =['accuracy'])


C:\ProgramData\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
# Model summary printing:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 46, 46, 50)          │             500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 23, 23, 50)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 21, 21, 64)          │          28,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 10, 10, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 8, 64)            │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 4, 4, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         262,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │           1,285 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 329,977 (1.26 MB)

 Trainable params: 329,977 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(xtrain,ytrain,epochs=15,validation_data=(xtest,ytest))


Epoch 1/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.3665 - loss: 1.4488 - val_accuracy: 0.4842 - val_loss: 1.2592
Epoch 2/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.5226 - loss: 1.1780 - val_accuracy: 0.5519 - val_loss: 1.1264
Epoch 3/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.5786 - loss: 1.0620 - val_accuracy: 0.5693 - val_loss: 1.0814
Epoch 4/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.6134 - loss: 0.9800 - val_accuracy: 0.5724 - val_loss: 1.0821
Epoch 5/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.6402 - loss: 0.9184 - val_accuracy: 0.5855 - val_loss: 1.0523
Epoch 6/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.6703 - loss: 0.8545 - val_accuracy: 0.5823 - val_loss: 1.0547
Epoch 7/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.6932 - loss: 0.7913 - val_accuracy: 0.5985 - val_loss: 1.0619
Epoch 8/15
453/453 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.7183 - loss: 0.7274 - 

In [14]:
# To save the model for deployment:
# .h5 -> HDF5 File format
# It stores model's optimizer's state(with learning rate), model's architecture, training configurtations, weights.
model.save('Emotion_detection_model_new.h5')
print("Model Saved")

Model Saved
